In [25]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser
import dotenv




In [26]:
dotenv.load_dotenv()

True

In [27]:
prompt1 = PromptTemplate(template="Generate a summary for {Topic}.",input_variables=["Topic"])


In [28]:
prompt2 = PromptTemplate(template="give the 5 quiz on this {Topic}",input_variables=["Topic"])

In [29]:
model1 = ChatGroq(model="llama-3.1-8b-instant")

In [30]:
model2 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [31]:
parser = StrOutputParser()


In [32]:
parallel_chain = RunnableParallel({'summary':prompt1 | model1 | parser,'quiz':prompt2 | model2 | parser})

In [33]:
prompt3 = PromptTemplate(template="Merge the following {summary} and {quiz} together",input_variables=['summary','quiz'])

In [35]:
chain3 = prompt3 | model1 | parser

In [36]:
final_chain = parallel_chain | chain3

In [37]:
result = final_chain.invoke({'Topic':"What is GenAi?"}) 

In [18]:
print(result)

**GenAi: A Comprehensive Summary**

GenAi, also known as General Artificial Intelligence (AGI), refers to a hypothetical artificial intelligence system that possesses the ability to understand, learn, and apply its knowledge across a wide range of tasks, much like a human being. This advanced AI system would be capable of reasoning, problem-solving, and adapting to new situations, enabling it to perform any intellectual task that a human can.

**Key Characteristics:**

1. **Autonomy**: GenAi systems would be able to operate independently, making decisions and taking actions without human intervention.
2. **Reasoning and Problem-Solving**: GenAi systems would be able to reason, draw conclusions, and solve complex problems using various techniques, such as logical reasoning, machine learning, and cognitive architectures.
3. **Knowledge Acquisition**: GenAi systems would be able to learn from a wide range of sources, including text, images, speech, and other forms of data.
4. **General In

In [104]:
final_chain.get_graph().print_ascii()

ImportError: Install grandalf to draw graphs: `pip install grandalf`.

### Conditional chain 



In [42]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableBranch,RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from typing import Literal
from pydantic import BaseModel,Field
import dotenv

In [98]:
class Sentiment(BaseModel):
    sentiment: Literal["Postive","Negative"]=Field(description="Generate the sentiment as either Positive or Negative")

In [99]:
parser = PydanticOutputParser(pydantic_object=Sentiment)

In [90]:
prompt1 = PromptTemplate(template="To generate sentiment for the following {feedback} in {format}",input_variables=['feedback'],partial_variables={"format": parser.get_format_instructions()})

In [91]:
model1 = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [86]:
chainS = prompt1 | model1 | parser

In [95]:
chainS.invoke({
    'feedback': 'I like the product'
}).sentiment

'Postive'

In [76]:
p1 = PromptTemplate(template="generate the positive response for the recieved {sentiment} to the user.",input_variables=['sentiment'])

In [77]:
p2= PromptTemplate(template="generate the sorry response for the recieved {sentiment} to the user.",input_variables=['sentiment'])

In [78]:
strparser = StrOutputParser()

In [100]:
conditional_chain = RunnableBranch((lambda x:x.sentiment == 'Positive',p1 | model1 | strparser),(lambda x:x.sentiment == 'Negative',p2 | model1 | strparser ),
        RunnableLambda(lambda x:'could not find the sentiment')       )

In [96]:
final_chain = chainS | conditional_chain

In [101]:
fin_res = final_chain.invoke({'feedback':'i dont like iphone it is hype only'})

In [103]:
fin_res

'Here are several options for a "sorry" response when the sentiment detected is \'Negative\', ranging from general to more specific, depending on the context of your interaction:\n\n**General & Empathetic:**\n\n1.  "I\'m sorry to hear that you\'re feeling negative. Please tell me more about what went wrong or how I can assist you better."\n2.  "I apologize if I haven\'t met your expectations. I\'ve detected a negative sentiment, and I\'d like to understand how I can make things right."\n3.  "Oh no, I\'m sorry to see that your sentiment is negative. How can I help turn things around?"\n4.  "I regret that your experience is not positive. My goal is to be helpful, so please let me know what\'s causing your frustration."\n\n**If the AI/System might be the cause:**\n\n5.  "I apologize if my previous response wasn\'t helpful or caused any frustration. I\'ve detected a negative sentiment. Could you please clarify what I misunderstood or what you\'re looking for?"\n6.  "It seems I\'ve missed t